In [ ]:
# CASMI 2026 v2 inference: retrieval (train U COCONUT) + multi-channel evidence + LightGBM reranker
import os, sys, glob, subprocess, time, shutil
T0 = time.time()
GENERATE = True
os.makedirs('/kaggle/working/numba_cache', exist_ok=True)
os.environ['NUMBA_CACHE_DIR'] = '/kaggle/working/numba_cache'
def find(pattern):
    hits = sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True), key=len)
    if not hits:
        raise FileNotFoundError(pattern)
    return hits[0]
tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
whl = [w for w in glob.glob('/kaggle/input/**/rdkit-*.whl', recursive=True) if tag in w]
print('rdkit wheel:', whl)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--no-deps', whl[0]], capture_output=True, text=True)
print(r.stdout[-500:], r.stderr[-500:])
import rdkit; print('rdkit', rdkit.__version__)
code_dir = os.path.dirname(os.path.dirname(find('casmi/engine.py')))
sys.path.insert(0, code_dir)
COMP = os.path.dirname(find('test.parquet'))
print('comp dir', COMP, 'code', code_dir)


In [ ]:
# work dir: pool + fingerprints from the dataset, spectrum cache rebuilt from train.parquet
os.makedirs('work', exist_ok=True)
for f in ['pool_meta.parquet', 'pool_fp.npy', 'pool_frag_off.npy', 'pool_frag_mass.npy', 'fp_bits.npy', 'train_structs.parquet', 'train_fp_sel.npy']:
    src = find(f)
    dst = f'work/{f}'
    if not os.path.exists(dst):
        os.symlink(src, dst)
from casmi.build import build_spec_cache
build_spec_cache(os.path.join(COMP, 'train.parquet'), 'work')
print(f'cache built {time.time()-T0:.0f}s')


In [ ]:
import numpy as np, pandas as pd, torch
sys.argv = ['x']
sys.path.insert(0, os.path.join(code_dir, 'scripts'))
from casmi.library import Library
from casmi.pool import Pool
from casmi.engine import Engine, EngineCfg
from casmi.ranker import RankerBag
from casmi import fpnet, chem
models = sorted(glob.glob('/kaggle/input/**/fpnet_*.pt', recursive=True))
rankers = sorted(glob.glob('/kaggle/input/**/ranker_*.pkl', recursive=True))
print('models', models, 'rankers', rankers, 'cuda', torch.cuda.is_available())
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
L = Library('work'); P = Pool('work')
tfp = np.load('work/train_fp_sel.npy')
bank = fpnet.ModelBank(models, device=dev)
xencs = sorted(glob.glob('/kaggle/input/**/xenc_*.pt', recursive=True))
xs = None
if xencs:
    from casmi import xenc
    xs = xenc.XScorer(models[0], xencs[0], device=dev)
print('xenc', xencs)
E = Engine(L, P, tfp, EngineCfg(generate=GENERATE), bank, xscorer=xs)
rb = RankerBag.load(rankers[0])
te = pd.read_parquet(os.path.join(COMP, 'test.parquet'))
te['nm'] = chem.neutral_mass(te.precursor_mz.values.astype(np.float64), te.adduct.values)
mols = list(te.groupby('molecule_id', sort=False))
rows, diag = [], []
for gi, (mid, sub) in enumerate(mols):
    nms = sub.nm.values[np.isfinite(sub.nm.values)]
    smis = []
    if len(nms):
        target = float(np.median(nms))
        spectra = []
        for r in sub.itertuples():
            ce = r.collision_energy_ev
            ce_arr = np.atleast_1d(np.asarray(ce, dtype=float)) if ce is not None else np.zeros(0)
            spectra.append(dict(mz=np.asarray(r.ms2_mzs, np.float64), it=np.asarray(r.ms2_normalized_intensities, np.float64),
                                mode=1 if r.ionization_mode == 'positive' else -1, adduct=r.adduct,
                                prec=float(r.precursor_mz), ce=float(ce_arr.mean()) if len(ce_arr) else np.nan,
                                ce_n=int(len(ce_arr)) if len(ce_arr) else 1))
        try:
            C, X, info = E.run(spectra, target)
            if len(C.pid):
                score = rb.predict(X)
                order = np.argsort(-score, kind='stable')
                seen = set()
                for i in order:
                    k = C.key[i]
                    if k in seen: continue
                    seen.add(k); smis.append(C.smiles[i])
                    if len(smis) >= 25: break
                diag.append(dict(molecule_id=mid, n_cand=len(C.pid), lib_max=info['lib_max'], top_sim=info['top_sim'], n_gen=info.get('n_gen', 0)))
        except Exception as e:
            print('ERROR', mid, repr(e))
    if not smis: smis = ['CCO']
    rows.append((mid, ';'.join(smis[:25])))
    if (gi + 1) % 25 == 0 or gi == len(mols) - 1:
        print(f'  {gi+1}/{len(mols)} molecules {time.time()-T0:.0f}s', flush=True)
sub = pd.DataFrame(rows, columns=['molecule_id', 'smiles'])
samp = pd.read_csv(os.path.join(COMP, 'sample_submission.csv'))
sub = samp[['molecule_id']].merge(sub, on='molecule_id', how='left')
sub['smiles'] = sub['smiles'].fillna('CCO')
assert len(sub) == len(samp) and sub.molecule_id.is_unique and sub.smiles.notna().all()
assert sub.smiles.map(lambda s: len(s.split(';'))).max() <= 25
sub.to_csv('submission.csv', index=False)
d = pd.DataFrame(diag); print(d.describe().to_string())
print('done', sub.shape, f'{time.time()-T0:.0f}s')
sub.head()
